# Capstone C — Drift Predictor: Recalibration Interval from Data
#
**Dataset (โปรดอ่านก่อนใช้งาน):**
- *NOAA GML globally-averaged atmospheric CO₂, monthly means* — Lan, X., Thoning, K.W., Dlugokencky, E.J.: *Trends in globally-averaged CO₂ determined from NOAA Global Monitoring Laboratory measurements*, Version 2026-09, https://gml.noaa.gov/ccgg/trends/ (U.S. Government work; fair credit required)
- ไฟล์: `datasets/co2_mm_gl.csv` — header `#`, sentinel `-99.99`, หน่วย ppm
- **แมปเข้าเคสเมโทรโลยี:** `decimal` → เวลานับจากการสอบเทียบล่าสุด · `average` → ค่าเบี่ยงเบนของ working standard เทียบ reference (ppm) · คำถามปลายทาง: **ควรตั้ง recalibration interval กี่เดือน**
#
## เป้าหมายของแทร็กนี้
#
จากซีรีส์ drift ยาว 47 ปี สร้างโมเดลพยากรณ์ที่ (1) วัดผลซื่อสัตย์ต่อเวลา, (2) เบียด naive baseline ได้, (3) ให้ **แถบความไม่แน่นอน** ที่แปลงเป็นการตัดสินใจเรื่องรอบสอบเทียบซ้ำได้ — พร้อมตอบผู้ตรวจว่าทำไม
#
*(ตัวอย่างเฉลยเต็มอยู่ใน `capstone_C_drift_predictor_solution.ipynb` — ทำของตัวเองก่อนเปิด)*

In [1]:
from pathlib import Path


def data_dir() -> Path:
    """หาโฟลเดอร์ datasets/ โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "datasets").exists():
            return p / "datasets"
    raise FileNotFoundError("ไม่พบ datasets/ — รันโน้ตบุ๊กจากใน repo")


DATA = data_dir()


## เตรียมข้อมูล (ให้มาแล้ว — รันผ่านก่อนทำข้อ)

In [2]:
import matplotlib.pyplot as plt  # noqa: E402
import polars as pl  # noqa: E402

# ฟอนต์ไทยสำหรับป้ายกราฟ — fallback list รองรับทุกแพลตฟอร์ม (macOS/Windows)
plt.rcParams["font.family"] = [
    "DejaVu Sans",
    "Noto Sans Thai",
    "Leelawadee UI",
    "Tahoma",
    "Thonburi",
]

raw = pl.read_csv(DATA / "co2_mm_gl.csv", comment_prefix="#")
df = raw.with_columns(
    pl.when(pl.col("average") < 0)
    .then(None)
    .otherwise(pl.col("average"))
    .alias("average")
).drop_nulls("average")
v = df["average"].to_numpy()
dec = df["decimal"].to_numpy()
N = len(v)
print(
    f"{N} เดือน | {dec[0]:.1f} → {dec[-1]:.1f} | {v[0]:.1f} → {v[-1]:.1f} ppm"
)

570 เดือน | 1979.0 → 2026.5 | 336.6 → 427.6 ppm


## ข้อ 1 — มองให้เห็นก่อนสร้างโมเดล
#
พลอตซีรีส์เต็ม แล้วถอดองค์ประกอบ 3 ส่วนด้วยมือ (trend: rolling mean 12 เดือน centered, seasonal: profile รายเดือน, residual) — รายงาน 3 ตัวเลข: อัตรา drift เฉลี่ย (ppm/ปี), amplitude ฤดูกาล (ppm), residual std (ppm)
#
- **TODO:** พลอตซีรีส์เต็ม
- **TODO:** `trend = rolling_mean(window_size=12, center=True, min_samples=6)`
- **TODO:** seasonal profile จาก detrended รายเดือน → พลอต 4 แผง (raw/trend/seasonal/residual)
- **TODO:** พิมพ์ 3 ตัวเลข

## ข้อ 2 — ไม้บรรทัด: naive baselines
#
คำนวณ MAE ของ persistence ($y_t = y_{t-1}$) และ seasonal naive ($y_t = y_{t-12}$) — โมเดลของคุณต้องชนะ **ทั้งสองตัว** จึงจะมีค่า
#
- **TODO:** `mean_absolute_error` ของสอง baseline — พิมพ์เก็บไว้เทียบท้ายงาน

## ข้อ 3 — โมเดลหลัก: lags + TimeSeriesSplit
#
สร้าง design matrix จาก lags (1, 12) → linear regression วัดด้วย `TimeSeriesSplit(n_splits=5)` — แล้วตอบใน markdown: ทำไมแบ่ง fold แบบสุ่มไม่ได้กับซีรีส์นี้ (คำตอบต้องพูดถึง trend)
#
- **TODO:** `X = column_stack([v[12:N-1], v[0:N-13]])`, `yv = v[13:]`
- **TODO:** วน `tscv` พิมพ์ MAE ต่อ fold + mean — เทียบ baseline

## ข้อ 4 — XGBoost บน level หรือบน difference?
#
รัน XGBoost สองแบบ: (ก) ทำนาย level ด้วย lags (1, 12) (ข) ทำนาย **difference** Δy = y_t − y_{t-12} ด้วย lags ของ Δ (แล้ว reconstruct ระดับ) — ตารางเทียบ 4 โมเดล (linear-level, linear-diff, xgb-level, xgb-diff) และตอบ: ทำไมต้นไม้แพ้บน level (เกี่ยวกับ extrapolation)
#
- **TODO:** แบบ (ก) — XGBRegressor บน `X`, `yv` ด้วย `tscv` (คาดหวังแย่มาก — บันทึกเป็นหลักฐาน)
- **TODO:** แบบ (ข) — `dy = v[12:] - v[:N-12]`; สร้าง `Xd` จาก lags ของ `dy` (`i0, i1 = 12, len(dy)`)
  - ทำนาย → reconstruct: `pred_level = v[12 + te] + pred_dy`
- **TODO:** ตารางสรุป 4 โมเดล (polars DataFrame)

## ข้อ 5 — พยากรณ์ + ตัดสินใจ recalibration interval
#
เลือกโมเดลผู้ชนะ วน recursive 24 เดือนพร้อมแถบ ±2σ·√h แล้วตอบ: ถ้า tolerance limit อยู่ที่ **+6 ppm** จากค่าล่าสุด (จูนให้ต่างจากตัวอย่าง) — recalibration interval ปลอดภัยกี่เดือน พร้อมพลอต: ข้อมูลท้าย + เส้นพยากรณ์ + แถบ + เส้น tolerance
#
- **TODO:** fit โมเดลผู้ชนะบนข้อมูลเต็ม
- **TODO:** recursive 24 ก้าว → `fc[]`, `band = 2*sigma*sqrt(1..24)`
- **TODO:** หา h แรกที่ `fc[h] + band[h]` เกิน tolerance → interval ปลอดภัย = จุดนั้นลบ margin
- **TODO:** พลอตครบชุด

## ข้อ 6 — สรุปส่งงาน
#
ย่อหน้า 3–5 บรรทัด: interval ที่แนะนำ, สมมุติฐานที่แนวคิดนี้พึ่งพา (drift เดินต่อแบบเดิม? ไม่มีการซ่อมบำรุงกลางทาง?), และอะไรจะทำให้คุณอยากตั้ง interval สั้นลงอีก